# Retrieval-Augmented Generation (RAG) with Milvus and LangChain

本指南演示如何使用 LangChain 和 Milvus 构建一个检索增强生成（RAG）系统。

RAG 系统结合了检索系统和生成模型，根据给定的提示生成新的文本。该系统首先使用 Milvus 从语料库中检索相关文档，然后利用生成模型基于这些检索到的文档生成新文本。

LangChain 是一种基于大语言模型（LLM）开发应用的框架。Milvus 是全球最先进的开源向量数据库，专为支持嵌入相似度搜索和人工智能应用而设计。

## Get started
## Prepare the data

我们使用 Langchain WebBaseLoader 从网络源加载文档，并通过 RecursiveCharacterTextSplitter 将其拆分为多个片段。

In [1]:
import os

CUSTOM_CACHE = r'F:\Teewon\Milvue\models'
os.environ['HF_HOME'] = CUSTOM_CACHE
os.environ['HF_HUB_CACHE'] = os.path.join(CUSTOM_CACHE, 'hub')
os.environ['TRANSFORMERS_CACHE'] = os.path.join(CUSTOM_CACHE, 'transformers')
os.environ['TORCH_HOME'] = CUSTOM_CACHE

In [2]:
import bs4
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create a WebBaseLoader instance to load documents from web sources
loader = WebBaseLoader(
    web_path=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content","post-title","post-header")
        ),# 只解析 HTML 中符合特定条件的部分
    )
)

# Load documents from web sources using the loader
documents=loader.load()

# Initialize a RecursiveCharacterTextSplitter for splitting text into chunks
text_spliiter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=0)

# Split the documents into chunks using the text_splitter
docs=text_spliiter.split_documents(documents)

# Inspect
docs[1]

C:\Users\Administrator\AppData\Local\Temp\ipykernel_27876\2095123501.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='Short-term memory: I would consider all the in-context learning (See Prompt Engineering) as utilizing short-term memory of the model to learn.\nLong-term memory: This provides the agent with the capability to retain and recall (infinite) information over extended periods, often by leveraging an external vector store and fast retrieval.\n\n\nTool use\n\nThe agent learns to call external APIs for extra information that is missing from the model weights (often hard to change after pre-training), including current information, code execution capability, access to proprietary information sources and more.\n\n\n\n\n\nOverview of a LLM-powered autonomous agent system.')

如我们所见，文档已经分割成多个片段，且数据内容涉及AI代理。

## 使用Milvus向量存储构建RAG链

我们将初始化一个Milvus向量存储，将文档加载到Milvus向量存储中，并在后台建立索引。

通过测试查询语句在Milvus向量存储中搜索文档，我们将获得前3条结果。

In [4]:
from rag_utils.vanilla import vectorstore

vectorstore.add_documents(docs)

query="What is self-reflection of an AI Agnet?"
vectorstore.similarity_search(query,3)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[Document(metadata={'pk': 5, 'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='Self-reflection is a vital aspect that allows autonomous agents to improve iteratively by refining past action decisions and correcting previous mistakes. It plays a crucial role in real-world tasks where trial and error are inevitable.\nReAct (Yao et al. 2023) integrates reasoning and acting within LLM by extending the action space to be a combination of task-specific discrete actions and the language space. The former enables LLM to interact with the environment (e.g. use Wikipedia search API), while the latter prompting LLM to generate reasoning traces in natural language.\nThe ReAct prompt template incorporates explicit steps for LLM to think, roughly formatted as:\nThought: ...\nAction: ...\nObservation: ...\n... (Repeated many times)'),
 Document(metadata={'pk': 7, 'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='The heuristic function dete

使用LCEL（LangChain表达式语言）构建RAG链。

In [6]:
from rag_utils.vanilla import format_docs,rag_prompt,llm
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Convert the vector store to a retriever检索器
retriever=vectorstore.as_retriever()

# Define the RAG
rag_chain=(
    {"context":retriever|format_docs, "question":RunnablePassthrough()}
    |rag_prompt
    |llm
    |StrOutputParser()
)

#rag_chain.get_graph().print_ascii()

res=rag_chain.invoke(query)
res

'Self-reflection in an AI agent is the process by which the agent critiques and learns from its past actions to improve future decisions. It refines past action decisions and corrects previous mistakes, which is important in real-world tasks where trial and error are inevitable.\n\nIn practice, self-reflection can be created by showing the LLM two-shot examples of failed trajectories paired with ideal reflections for guiding future plan changes. These reflections are then added to the agent’s working memory—up to three—and used as context for future LLM queries. A related reflection mechanism also synthesizes memories into higher-level summaries of past events to guide future behavior.'